# [LAB-04] 데이터 재구조화

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
from jussam import load_data
from pandas import DataFrame, pivot_table, melt

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 피벗 테이블 실습 데이터 가져오기

In [2]:
origin = load_data("city_people")
origin

📚 서울, 인천, 부산에서 2005년, 2010년 2015년에 조사한 가상의 인구수 데이터(인덱스와 메타데이터 없음)


,도시,연도,인구,지역
0,서울,2015,9904312,수도권
1,서울,2010,9631482,수도권
2,서울,2005,9762546,수도권
3,부산,2015,3448737,경상권
4,부산,2010,3393191,경상권
5,부산,2005,3512547,경상권
6,인천,2015,2890451,수도권
7,인천,2010,2632035,수도권


## #02. 피벗 테이블

### 1. 기본 사용 방법 (도시와 연도에 따른 인구수 재배치)

- 인덱스, 컬럼, 값으로 사용할 필드를 각각 지정하여 데이터를 재배치한다.
- 리턴되는 피벗 테이블 역시 데이터 프레임 객체이다.

In [3]:
pivot1 = pivot_table(origin,            # 피벗할 데이터프레임
                     index = '도시',     # 행 위치에 들어갈 열
                     columns = '연도',   # 열 위치에 들어갈 열
                     values = '인구'     # 데이터로 사용할 열
)
print(type(pivot1))
pivot1

<class 'pandas.core.frame.DataFrame'>


연도,2005,2010,2015
도시,,,
부산,3512547.000,3393191.000,3448737.000
서울,9762546.000,9631482.000,9904312.000
인천,NaN,2632035.000,2890451.000


### 2. 중복 데이터의 집계 방법 지정하기

- 인덱스와 컬럼에 따른 value가 두 개 이상인 경우 집계 방법을 지정해야 정확한 결과를 얻을 수 있다.
  - 예제에서는 2010년과 2015년에 대한 수도권 데이터가 두 개씩 존재한다.
  - 전체 인구수를 얻어야 하는 경우라면 합계를 구해야 한다.

In [4]:
pivot_table(origin,          # 피벗할 데이터프레임
            index = '지역',   # 행 위치에 들어갈 열
            columns = '연도', # 열 위치에 들어갈 열
            values = '인구',  # 데이터로 사용할 열
            aggfunc = 'sum'  # 데이터가 두 개 이상일 경우 집계함수 지정
)

연도,2005,2010,2015
지역,,,
경상권,3512547,3393191,3448737
수도권,9762546,12263517,12794763


### 3. 두 개 이상의 집계 함수 지정

- 집계 함수의 이름을 리스트로 설정한다.

In [5]:
pivot_table(origin,
            index = '지역',
            columns = '연도',
            values = '인구',
            aggfunc = ['sum', 'mean']
)

sum                            mean                        
연도      2005      2010      2015        2005        2010        2015
지역                                                                  
경상권  3512547   3393191   3448737 3512547.000 3393191.000 3448737.000
수도권  9762546  12263517  12794763 9762546.000 6131758.500 6397381.500

### 4. 복수 인덱스 지정

In [6]:
pivot_table(origin,
            index = ['지역', '연도'],
            columns = '도시',
            values = '인구',
            aggfunc =  ['mean', 'sum']
)

mean                                 sum              \
도시                부산          서울          인천          부산          서울   
지역  연도                                                                 
경상권 2005 3512547.000         NaN         NaN 3512547.000         NaN   
    2010 3393191.000         NaN         NaN 3393191.000         NaN   
    2015 3448737.000         NaN         NaN 3448737.000         NaN   
수도권 2005         NaN 9762546.000         NaN         NaN 9762546.000   
    2010         NaN 9631482.000 2632035.000         NaN 9631482.000   
    2015         NaN 9904312.000 2890451.000         NaN 9904312.000   

                      
도시                인천  
지역  연도                
경상권 2005         NaN  
    2010         NaN  
    2015         NaN  
수도권 2005         NaN  
    2010 2632035.000  
    2015 2890451.000

## #03. melt

### 1. 샘플 피벗 테이블 생성

In [7]:
pivot_df = pivot_table(origin,
                       index = '연도', columns = '지역',
                       values = '인구', aggfunc='mean')
pivot_df

지역,경상권,수도권
연도,,
2005,3512547.000,9762546.000
2010,3393191.000,6131758.500
2015,3448737.000,6397381.500


### 2. 피벗 테이블 분리 1단계

- 데이터프레임의 인덱스를 일반 컬럼으로 설정한다.

In [8]:
pivot_df2 = pivot_df.reset_index()
pivot_df2

지역,연도,경상권,수도권
0,2005,3512547.000,9762546.000
1,2010,3393191.000,6131758.500
2,2015,3448737.000,6397381.500


### 3. 피벗 테이블 분리 2단계 : melt 처리

- id_vars : 인덱스로 사용할 컬럼이름. 반드시 컬럼만 가능(인덱스 불가)
- value_vars: 분리할 컬럼 이름들

In [9]:
mdf = melt(pivot_df2, id_vars=['연도'],
                    value_vars=['경상권', '수도권'])
mdf

,연도,지역,value
0,2005,경상권,3512547.000
1,2010,경상권,3393191.000
2,2015,경상권,3448737.000
3,2005,수도권,9762546.000
4,2010,수도권,6131758.500
5,2015,수도권,6397381.500


### 4. 피벗 테이블 분리시 필드 이름 지정하기

In [10]:
mdf = melt(pivot_df2, id_vars=['연도'],
           value_vars=['경상권', '수도권'],
           var_name='구분', value_name='인구수')
mdf

,연도,구분,인구수
0,2005,경상권,3512547.000
1,2010,경상권,3393191.000
2,2015,경상권,3448737.000
3,2005,수도권,9762546.000
4,2010,수도권,6131758.500
5,2015,수도권,6397381.500


## #04. Stack, Unstack 실습을 위한 준비작업

### 1. 샘플 데이터 가져오기

In [11]:
origin = load_data("body_size")
origin

📚 어느 학급 학생들의 이름, 성별, 키, 몸무게에 대한 가상의 데이터

    field    description
--  -------  -------------
 0  name     이름
 1  sex      성별
 2  height   키
 3  weight   몸무게



,sex,height,weight
name,,,
Lee,M,175,98.000
Park,F,167,48.000
Hong,M,180,NaN
Kim,F,162,55.000
Nam,M,172,85.000


## #05. stack

### 1. 기본 사용 방법

- 인덱스 열에 따라 모든 변수를 하나의 변수로 쌓아놓는 처리로 리턴 결과는 Series 객체가 된다.
- 원본 데이터프레임의 컬럼이름이 카테고리 역할을 하는 변수로 배치되고 이에 따라 원본의 값이 재배치 된다.
- **빈 값(`NaN`)은 stack 과정에서 제거된다.**
  - 예제에서는 `Hong`의 `Weight`가 제외되어 있음

In [12]:
st1 = origin.stack()
print(type(st1))
st1

<class 'pandas.core.series.Series'>


name        
Lee   sex           M
      height      175
      weight   98.000
Park  sex           F
      height      167
      weight   48.000
Hong  sex           M
      height      180
Kim   sex           F
      height      162
      weight   55.000
Nam   sex           M
      height      172
      weight   85.000
dtype: object

### 2. Stack 결과를 DataFrame으로 생성하기

In [13]:
df1 = DataFrame(origin.stack())
df1

0
name              
Lee  sex         M
     height    175
     weight 98.000
Park sex         F
     height    167
     weight 48.000
Hong sex         M
     height    180
Kim  sex         F
     height    162
     weight 55.000
Nam  sex         M
     height    172
     weight 85.000

## #06. Unstack

### 1. stack 결과를 원래대로 되돌린다.

In [14]:
ust = df1.unstack()
ust

0              
     sex height weight
name                  
Lee    M    175 98.000
Park   F    167 48.000
Hong   M    180    NaN
Kim    F    162 55.000
Nam    M    172 85.000